In [2]:
from pathlib import Path
import sys
import json
import random
import time

import numpy as np
import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:")
print(PROJECT_ROOT)

Project root:
c:\Work\Quantum-Adversarial-Robustness


In [3]:
SEED = 42


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(SEED)

print("Seed:", SEED)

Seed: 42


In [4]:
NUM_QUBITS = 4

DEVICE = torch.device("cpu")

MODEL_PATH = (
    PROJECT_ROOT
    / "results"
    / "models"
    / "vqc_04B_scaled_seed42.pt"
)

SCALER_PATH = (
    PROJECT_ROOT
    / "results"
    / "preprocessing"
    / "standard_scaler_04B_seed42.joblib"
)

DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "binary"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "adversarial"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 60)
print("NOTEBOOK 05 — FGSM ATTACK")
print("=" * 60)

print("Model :", MODEL_PATH)
print("Scaler:", SCALER_PATH)
print("Device:", DEVICE)

NOTEBOOK 05 — FGSM ATTACK
Model : c:\Work\Quantum-Adversarial-Robustness\results\models\vqc_04B_scaled_seed42.pt
Scaler: c:\Work\Quantum-Adversarial-Robustness\results\preprocessing\standard_scaler_04B_seed42.joblib
Device: cpu


In [5]:
assert MODEL_PATH.exists(), (
    f"Model checkpoint not found:\n{MODEL_PATH}"
)

assert SCALER_PATH.exists(), (
    f"Scaler not found:\n{SCALER_PATH}"
)

print("Model checkpoint: FOUND")
print("Scaler:           FOUND")

Model checkpoint: FOUND
Scaler:           FOUND


In [6]:
X_train = np.load(
    DATA_DIR / "X_train.npy"
)

X_test = np.load(
    DATA_DIR / "X_test.npy"
)

y_train = np.load(
    DATA_DIR / "y_train.npy"
)

y_test = np.load(
    DATA_DIR / "y_test.npy"
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (11824, 4)
X_test : (2956, 4)
y_train: (11824,)
y_test : (2956,)


In [7]:
import joblib

scaler = joblib.load(
    SCALER_PATH
)

print(scaler)

StandardScaler()


In [8]:
X_test_scaled = scaler.transform(
    X_test
)

print("Scaled test shape:")
print(X_test_scaled.shape)

print()
print("Scaled test statistics:")

print(
    "Min :",
    X_test_scaled.min()
)

print(
    "Max :",
    X_test_scaled.max()
)

print(
    "Mean:",
    X_test_scaled.mean()
)

print(
    "Std :",
    X_test_scaled.std()
)

Scaled test shape:
(2956, 4)

Scaled test statistics:
Min : -3.198715518689433
Max : 2.896355402745653
Mean: 0.013355479891455179
Std : 0.9977721869248157


In [9]:
X_test_tensor = torch.tensor(
    X_test_scaled,
    dtype=torch.float32,
    device=DEVICE
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.float32,
    device=DEVICE
).reshape(-1, 1)

print("X_test_tensor:")
print(X_test_tensor.shape)

print()

print("y_test_tensor:")
print(y_test_tensor.shape)

X_test_tensor:
torch.Size([2956, 4])

y_test_tensor:
torch.Size([2956, 1])


In [10]:
from src.models.quantum_model import create_model
from src.models.hybrid_classifier import HybridClassifier
model = create_model(
    num_qubits=4,
    seed=42
)

print(model)

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


TorchConnector()


In [12]:
quantum_model = create_model(
    num_qubits=NUM_QUBITS
)

model = HybridClassifier(
    quantum_model=quantum_model
)

model = model.to(DEVICE)

print(model)

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


HybridClassifier(
  (quantum): TorchConnector()
  (classifier): Linear(in_features=1, out_features=1, bias=True)
)


In [13]:
state_dict = torch.load(
    MODEL_PATH,
    map_location=DEVICE
)

model.load_state_dict(
    state_dict
)

model.eval()

print("Successfully loaded trained 04B model.")

Successfully loaded trained 04B model.


In [14]:
print("=" * 60)
print("TRAINED MODEL PARAMETERS")
print("=" * 60)

print(
    "Classifier weight:"
)

print(
    model.classifier.weight.detach()
)

print()

print(
    "Classifier bias:"
)

print(
    model.classifier.bias.detach()
)

TRAINED MODEL PARAMETERS
Classifier weight:
tensor([[2.0504]])

Classifier bias:
tensor([0.0436])


In [15]:
with torch.no_grad():

    clean_logits = model(
        X_test_tensor
    )

    clean_probabilities = torch.sigmoid(
        clean_logits
    )

    clean_predictions = (
        clean_probabilities >= 0.5
    ).int()

In [ ]:
y_true = y_test_tensor.cpu().numpy().reshape(-1)
y_clean = clean_predictions.cpu().numpy().reshape(-1)

clean_accuracy = accuracy_score(
    y_true,
    y_clean
)

clean_precision = precision_score(
    y_true,
    y_clean,
    zero_division=0
)

clean_recall = recall_score(
    y_true,
    y_clean,
    zero_division=0
)

clean_f1 = f1_score(
    y_true,
    y_clean,
    zero_division=0
)

clean_cm = confusion_matrix(
    y_true,
    y_clean
)

print("=" * 60)
print("CLEAN 04B BASELINE")
print("=" * 60)

print(f"Accuracy : {clean_accuracy:.4f}")
print(f"Precision: {clean_precision:.4f}")
print(f"Recall   : {clean_recall:.4f}")
print(f"F1       : {clean_f1:.4f}")

print()
print("Confusion matrix:")
print(clean_cm)

In [ ]:
def fgsm_attack(
    model,
    x,
    y,
    epsilon,
    criterion,
):
    """
    Generate an FGSM adversarial example.

    Parameters
    ----------
    model : torch.nn.Module
        Trained hybrid quantum-classical model.

    x : torch.Tensor
        Input features.

    y : torch.Tensor
        True labels.

    epsilon : float
        FGSM perturbation magnitude.

    criterion : torch.nn.Module
        Classification loss.

    Returns
    -------
    x_adv : torch.Tensor
        Adversarial examples.

    perturbation : torch.Tensor
        Applied perturbation.
    """

    x_adv = x.clone().detach()

    x_adv.requires_grad = True

    model.zero_grad(set_to_none=True)

    logits = model(
        x_adv
    )

    loss = criterion(
        logits,
        y
    )

    loss.backward()

    gradient = x_adv.grad.detach()

    perturbation = (
        epsilon
        * gradient.sign()
    )

    x_adv = (
        x_adv.detach()
        + perturbation
    )

    return x_adv, perturbation

In [ ]:
criterion = nn.BCEWithLogitsLoss()

print(criterion)

In [ ]:
TEST_BATCH_SIZE = 32

x_test_batch = X_test_tensor[
    :TEST_BATCH_SIZE
]

y_test_batch = y_test_tensor[
    :TEST_BATCH_SIZE
]

epsilon_test = 0.10

x_adv_batch, perturbation_batch = fgsm_attack(
    model=model,
    x=x_test_batch,
    y=y_test_batch,
    epsilon=epsilon_test,
    criterion=criterion,
)

print("Original shape:")
print(x_test_batch.shape)

print()

print("Adversarial shape:")
print(x_adv_batch.shape)

print()

print("Perturbation shape:")
print(perturbation_batch.shape)

In [ ]:
actual_perturbation = (
    x_adv_batch
    - x_test_batch
)

print("=" * 60)
print("FGSM PERTURBATION CHECK")
print("=" * 60)

print(
    "Maximum absolute perturbation:",
    actual_perturbation.abs().max().item()
)

print(
    "Requested epsilon:",
    epsilon_test
)

In [ ]:
with torch.no_grad():

    clean_batch_logits = model(
        x_test_batch
    )

    adv_batch_logits = model(
        x_adv_batch
    )

    clean_batch_predictions = (
        torch.sigmoid(clean_batch_logits)
        >= 0.5
    ).int()

    adv_batch_predictions = (
        torch.sigmoid(adv_batch_logits)
        >= 0.5
    ).int()

changed = (
    clean_batch_predictions
    != adv_batch_predictions
).sum().item()

print(
    "Prediction changes:",
    changed,
    "/",
    TEST_BATCH_SIZE
)

In [ ]:
# ============================================================
# FIXED CLEAN REFERENCE
# ============================================================

model.eval()

with torch.no_grad():
    clean_logits = model(X_test_tensor)
    clean_probabilities = torch.sigmoid(clean_logits)
    clean_predictions = (
        clean_probabilities >= 0.5
    ).int()

y_true = (
    y_test_tensor
    .detach()
    .cpu()
    .numpy()
    .reshape(-1)
)

clean_pred_np = (
    clean_predictions
    .detach()
    .cpu()
    .numpy()
    .reshape(-1)
)

clean_logits_np = (
    clean_logits
    .detach()
    .cpu()
    .numpy()
    .reshape(-1)
)

clean_prob_np = (
    clean_probabilities
    .detach()
    .cpu()
    .numpy()
    .reshape(-1)
)

clean_accuracy = accuracy_score(
    y_true,
    clean_pred_np
)

clean_precision = precision_score(
    y_true,
    clean_pred_np,
    zero_division=0
)

clean_recall = recall_score(
    y_true,
    clean_pred_np,
    zero_division=0
)

clean_f1 = f1_score(
    y_true,
    clean_pred_np,
    zero_division=0
)

clean_cm = confusion_matrix(
    y_true,
    clean_pred_np
)

clean_correct = (
    clean_pred_np == y_true
)

print("=" * 60)
print("FIXED CLEAN 04B REFERENCE")
print("=" * 60)

print(f"Accuracy : {clean_accuracy:.6f}")
print(f"Precision: {clean_precision:.6f}")
print(f"Recall   : {clean_recall:.6f}")
print(f"F1       : {clean_f1:.6f}")

print()
print("Correctly classified:", clean_correct.sum())
print("Total samples:", len(y_true))

print()
print("Confusion matrix:")
print(clean_cm)

In [ ]:
def evaluate_fgsm_fixed_reference(
    model,
    X,
    y,
    clean_predictions,
    clean_correct,
    epsilon,
    criterion,
):
    """
    Evaluate FGSM against a fixed clean reference.

    The clean predictions are computed once outside this
    function. This guarantees that accuracy drop and ASR
    are calculated relative to the same clean model output
    for every epsilon.
    """

    model.eval()

    # --------------------------------------------------------
    # Generate adversarial examples
    # --------------------------------------------------------

    x_adv, perturbation = fgsm_attack(
        model=model,
        x=X,
        y=y,
        epsilon=epsilon,
        criterion=criterion,
    )

    # --------------------------------------------------------
    # Evaluate adversarial examples
    # --------------------------------------------------------

    with torch.no_grad():

        adv_logits = model(x_adv)

        adv_probabilities = torch.sigmoid(
            adv_logits
        )

        adv_predictions = (
            adv_probabilities >= 0.5
        ).int()

    y_np = (
        y.detach()
        .cpu()
        .numpy()
        .reshape(-1)
    )

    adv_pred_np = (
        adv_predictions.detach()
        .cpu()
        .numpy()
        .reshape(-1)
    )

    # --------------------------------------------------------
    # Adversarial classification metrics
    # --------------------------------------------------------

    adv_accuracy = accuracy_score(
        y_np,
        adv_pred_np
    )

    adv_precision = precision_score(
        y_np,
        adv_pred_np,
        zero_division=0
    )

    adv_recall = recall_score(
        y_np,
        adv_pred_np,
        zero_division=0
    )

    adv_f1 = f1_score(
        y_np,
        adv_pred_np,
        zero_division=0
    )

    # --------------------------------------------------------
    # Accuracy degradation
    # --------------------------------------------------------

    accuracy_drop = (
        clean_accuracy
        - adv_accuracy
    )

    # --------------------------------------------------------
    # Prediction changes
    # --------------------------------------------------------

    prediction_changed = (
        clean_predictions.detach()
        .cpu()
        .numpy()
        .reshape(-1)
        != adv_pred_np
    )

    prediction_change_rate = (
        prediction_changed.mean()
    )

    # --------------------------------------------------------
    # Attack success rate
    #
    # Only samples that were correctly classified by
    # the clean model are considered.
    # --------------------------------------------------------

    successful_attack = (
        clean_correct
        & (adv_pred_np != y_np)
    )

    clean_correct_samples = (
        clean_correct.sum()
    )

    successful_attacks = (
        successful_attack.sum()
    )

    if clean_correct_samples > 0:

        attack_success_rate = (
            successful_attacks
            / clean_correct_samples
        )

    else:

        attack_success_rate = 0.0

    # --------------------------------------------------------
    # Perturbation statistics
    # --------------------------------------------------------

    perturbation_np = (
        perturbation.detach()
        .cpu()
        .numpy()
    )

    l_inf = np.max(
        np.abs(perturbation_np),
        axis=1
    )

    l_2 = np.linalg.norm(
        perturbation_np,
        axis=1
    )

    return {
        "epsilon": float(epsilon),

        "clean_accuracy": float(
            clean_accuracy
        ),

        "clean_precision": float(
            clean_precision
        ),

        "clean_recall": float(
            clean_recall
        ),

        "clean_f1": float(
            clean_f1
        ),

        "adversarial_accuracy": float(
            adv_accuracy
        ),

        "adversarial_precision": float(
            adv_precision
        ),

        "adversarial_recall": float(
            adv_recall
        ),

        "adversarial_f1": float(
            adv_f1
        ),

        "accuracy_drop": float(
            accuracy_drop
        ),

        "prediction_change_rate": float(
            prediction_change_rate
        ),

        "attack_success_rate": float(
            attack_success_rate
        ),

        "clean_correct_samples": int(
            clean_correct_samples
        ),

        "successful_attacks": int(
            successful_attacks
        ),

        "mean_l2_perturbation": float(
            l_2.mean()
        ),

        "max_l2_perturbation": float(
            l_2.max()
        ),

        "mean_linf_perturbation": float(
            l_inf.mean()
        ),

        "max_linf_perturbation": float(
            l_inf.max()
        ),
    }

In [ ]:
zero_result = evaluate_fgsm_fixed_reference(
    model=model,
    X=X_test_tensor,
    y=y_test_tensor,
    clean_predictions=clean_predictions,
    clean_correct=clean_correct,
    epsilon=0.0,
    criterion=criterion,
)

print("=" * 60)
print("FGSM SANITY CHECK — EPSILON = 0")
print("=" * 60)

print(
    "Clean accuracy:",
    zero_result["clean_accuracy"]
)

print(
    "Adversarial accuracy:",
    zero_result["adversarial_accuracy"]
)

print(
    "Accuracy drop:",
    zero_result["accuracy_drop"]
)

print(
    "Prediction change rate:",
    zero_result["prediction_change_rate"]
)

print(
    "Attack success rate:",
    zero_result["attack_success_rate"]
)

In [ ]:
EPSILONS = [
    0.01,
    0.05,
    0.10,
    0.20,
    0.30,
]

fgsm_results = []

for epsilon in EPSILONS:

    print()
    print("=" * 60)
    print(f"FGSM EXPERIMENT — epsilon = {epsilon}")
    print("=" * 60)

    start_time = time.perf_counter()

    result = evaluate_fgsm_fixed_reference(
        model=model,
        X=X_test_tensor,
        y=y_test_tensor,
        clean_predictions=clean_predictions,
        clean_correct=clean_correct,
        epsilon=epsilon,
        criterion=criterion,
    )

    elapsed = (
        time.perf_counter()
        - start_time
    )

    result["runtime_seconds"] = float(
        elapsed
    )

    fgsm_results.append(result)

    print(
        f"Clean accuracy : "
        f"{result['clean_accuracy']:.4f}"
    )

    print(
        f"Adv accuracy   : "
        f"{result['adversarial_accuracy']:.4f}"
    )

    print(
        f"Accuracy drop  : "
        f"{result['accuracy_drop']:.4f}"
    )

    print(
        f"Adv F1         : "
        f"{result['adversarial_f1']:.4f}"
    )

    print(
        f"Attack success : "
        f"{result['attack_success_rate']:.4f}"
    )

    print(
        f"Runtime        : "
        f"{elapsed / 60:.2f} minutes"
    )

In [ ]:
import pandas as pd

fgsm_df = pd.DataFrame(
    fgsm_results
)

columns = [
    "epsilon",
    "clean_accuracy",
    "adversarial_accuracy",
    "accuracy_drop",
    "adversarial_precision",
    "adversarial_recall",
    "adversarial_f1",
    "attack_success_rate",
    "prediction_change_rate",
    "mean_l2_perturbation",
    "mean_linf_perturbation",
]

display(
    fgsm_df[columns].round(4)
)

In [ ]:
CSV_PATH = (
    RESULTS_DIR
    / "fgsm_04B_seed42.csv"
)

fgsm_df.to_csv(
    CSV_PATH,
    index=False
)

print("CSV saved:")
print(CSV_PATH)

In [ ]:
FGSM_JSON_PATH = (
    RESULTS_DIR
    / "fgsm_04B_seed42.json"
)

experiment_metadata = {
    "experiment": "05_FGSM",
    "base_model": "04B",
    "seed": SEED,
    "num_qubits": NUM_QUBITS,
    "dataset": "Binary MNIST 0 vs 1",
    "input_features": 4,
    "preprocessing": "PCA + StandardScaler",
    "attack": "FGSM",
    "loss": "BCEWithLogitsLoss",
    "epsilon_space": "standardized PCA feature space",
    "results": fgsm_results,
}

with open(
    FGSM_JSON_PATH,
    "w"
) as f:

    json.dump(
        experiment_metadata,
        f,
        indent=4
    )

print("JSON saved:")
print(FGSM_JSON_PATH)

In [ ]:
CLEAN_REFERENCE_PATH = (
    RESULTS_DIR
    / "clean_reference_04B_seed42.json"
)

clean_reference = {
    "experiment": "04B_clean_reference",
    "seed": SEED,
    "num_qubits": NUM_QUBITS,
    "accuracy": float(clean_accuracy),
    "precision": float(clean_precision),
    "recall": float(clean_recall),
    "f1": float(clean_f1),
    "clean_correct_samples": int(
        clean_correct.sum()
    ),
    "total_test_samples": int(
        len(y_true)
    ),
}

with open(
    CLEAN_REFERENCE_PATH,
    "w"
) as f:

    json.dump(
        clean_reference,
        f,
        indent=4
    )

print("Clean reference saved:")
print(CLEAN_REFERENCE_PATH)

In [ ]:
PREDICTION_PATH = (
    RESULTS_DIR
    / "clean_predictions_04B_seed42.npy"
)

np.save(
    PREDICTION_PATH,
    clean_pred_np
)

print("Clean predictions saved:")
print(PREDICTION_PATH)

In [ ]:
import matplotlib.pyplot as plt

eps = fgsm_df["epsilon"].to_numpy()

adv_accuracy = (
    fgsm_df[
        "adversarial_accuracy"
    ].to_numpy()
)

plt.figure(figsize=(8, 5))

plt.plot(
    eps,
    adv_accuracy,
    marker="o",
    linewidth=2,
    label="FGSM accuracy"
)

plt.axhline(
    clean_accuracy,
    linestyle="--",
    linewidth=1.5,
    label="Clean accuracy"
)

plt.xlabel(
    "FGSM perturbation ε"
)

plt.ylabel(
    "Test accuracy"
)

plt.title(
    "VQC Accuracy Under FGSM Attack"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "fgsm_accuracy_vs_epsilon.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
accuracy_drop = (
    fgsm_df[
        "accuracy_drop"
    ].to_numpy()
)

plt.figure(figsize=(8, 5))

plt.plot(
    eps,
    accuracy_drop,
    marker="o",
    linewidth=2
)

plt.xlabel(
    "FGSM perturbation ε"
)

plt.ylabel(
    "Accuracy degradation"
)

plt.title(
    "Accuracy Degradation Under FGSM"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "fgsm_accuracy_drop_vs_epsilon.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
asr = (
    fgsm_df[
        "attack_success_rate"
    ].to_numpy()
)

plt.figure(figsize=(8, 5))

plt.plot(
    eps,
    asr,
    marker="o",
    linewidth=2
)

plt.xlabel(
    "FGSM perturbation ε"
)

plt.ylabel(
    "Attack Success Rate"
)

plt.title(
    "FGSM Attack Success Rate"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "fgsm_attack_success_rate.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
f1_values = (
    fgsm_df[
        "adversarial_f1"
    ].to_numpy()
)

plt.figure(figsize=(8, 5))

plt.plot(
    eps,
    f1_values,
    marker="o",
    linewidth=2
)

plt.xlabel(
    "FGSM perturbation ε"
)

plt.ylabel(
    "F1 score"
)

plt.title(
    "VQC F1 Score Under FGSM Attack"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "fgsm_f1_vs_epsilon.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
mean_linf = (
    fgsm_df[
        "mean_linf_perturbation"
    ].to_numpy()
)

mean_l2 = (
    fgsm_df[
        "mean_l2_perturbation"
    ].to_numpy()
)

plt.figure(figsize=(8, 5))

plt.plot(
    eps,
    mean_linf,
    marker="o",
    linewidth=2,
    label="Mean L∞"
)

plt.plot(
    eps,
    mean_l2,
    marker="s",
    linewidth=2,
    label="Mean L2"
)

plt.xlabel(
    "FGSM perturbation ε"
)

plt.ylabel(
    "Perturbation magnitude"
)

plt.title(
    "FGSM Perturbation Magnitude"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "fgsm_perturbation_magnitude.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
print("=" * 70)
print("NOTEBOOK 05 — FINAL FGSM EXPERIMENT")
print("=" * 70)

print()
print("Base model:")
print("  04B — Standardized PCA VQC")

print()
print("Dataset:")
print("  Binary MNIST — digits 0 vs 1")

print()
print("Quantum model:")
print("  4 qubits")
print("  ZZFeatureMap reps=2")
print("  RealAmplitudes reps=2")
print("  Linear entanglement")

print()
print("Clean reference:")
print(f"  Accuracy : {clean_accuracy:.4f}")
print(f"  Precision: {clean_precision:.4f}")
print(f"  Recall   : {clean_recall:.4f}")
print(f"  F1       : {clean_f1:.4f}")

print()
print("FGSM epsilon values:")
print(EPSILONS)

print()
print("Results:")
display(
    fgsm_df[
        [
            "epsilon",
            "adversarial_accuracy",
            "accuracy_drop",
            "adversarial_f1",
            "attack_success_rate",
        ]
    ].round(4)
)

print()
print("Results directory:")
print(RESULTS_DIR)

print()
print("=" * 70)

In [ ]:
# ============================================================
# DIAGNOSTIC 1 — REPEATED FORWARD PASSES
# ============================================================

model.eval()

x_diag = X_test_tensor[:100].clone().detach()

with torch.no_grad():
    logits_1 = model(x_diag)
    logits_2 = model(x_diag)
    logits_3 = model(x_diag)

logits_1 = logits_1.detach().cpu()
logits_2 = logits_2.detach().cpu()
logits_3 = logits_3.detach().cpu()

print("=" * 60)
print("REPEATED FORWARD PASS DIAGNOSTIC")
print("=" * 60)

print(
    "Max difference 1 -> 2:",
    torch.max(torch.abs(logits_1 - logits_2)).item()
)

print(
    "Max difference 2 -> 3:",
    torch.max(torch.abs(logits_2 - logits_3)).item()
)

print(
    "Predictions 1 == 2:",
    torch.equal(
        (logits_1 >= 0),
        (logits_2 >= 0)
    )
)

print(
    "Predictions 2 == 3:",
    torch.equal(
        (logits_2 >= 0),
        (logits_3 >= 0)
    )
)

In [ ]:
# ============================================================
# DIAGNOSTIC 2 — FORWARD -> BACKWARD -> FORWARD
# ============================================================

model.eval()

x_diag = X_test_tensor[:100].clone().detach()
y_diag = y_test_tensor[:100].clone().detach()

# First forward
model.zero_grad(set_to_none=True)

with torch.no_grad():
    logits_before = model(x_diag)

# Forward + backward
x_grad = x_diag.clone().detach()
x_grad.requires_grad_(True)

model.zero_grad(set_to_none=True)

logits_grad = model(x_grad)

loss_diag = criterion(
    logits_grad,
    y_diag
)

loss_diag.backward()

# Second forward
model.zero_grad(set_to_none=True)

with torch.no_grad():
    logits_after = model(x_diag)

logits_before = logits_before.detach().cpu()
logits_after = logits_after.detach().cpu()

print("=" * 60)
print("FORWARD -> BACKWARD -> FORWARD")
print("=" * 60)

print(
    "Loss:",
    loss_diag.item()
)

print(
    "Max logit difference:",
    torch.max(
        torch.abs(
            logits_before - logits_after
        )
    ).item()
)

print(
    "Predictions identical:",
    torch.equal(
        logits_before >= 0,
        logits_after >= 0
    )
)

In [ ]:
# ============================================================
# DIAGNOSTIC 3 — PARAMETER INTEGRITY
# ============================================================

model.eval()

params_before = {
    name: param.detach().clone()
    for name, param in model.named_parameters()
}

x_diag = X_test_tensor[:32].clone().detach()
y_diag = y_test_tensor[:32].clone().detach()

x_diag.requires_grad_(True)

model.zero_grad(set_to_none=True)

logits = model(x_diag)

loss = criterion(
    logits,
    y_diag
)

loss.backward()

params_after = {
    name: param.detach().clone()
    for name, param in model.named_parameters()
}

print("=" * 60)
print("PARAMETER INTEGRITY CHECK")
print("=" * 60)

for name in params_before:

    difference = torch.max(
        torch.abs(
            params_before[name]
            - params_after[name]
        )
    ).item()

    print(
        f"{name}: max change = {difference}"
    )

In [ ]:
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator

from src.models.quantum_model import (
    create_quantum_circuit,
    create_qnn,
)
estimator_test = StatevectorEstimator(
    seed=42
)

print(estimator_test)

# ============================================================
# DIRECT QISKIT ESTIMATOR REPRODUCIBILITY TEST
# ============================================================

qc, feature_map, ansatz = create_quantum_circuit(
    num_qubits=4
)

observable = SparsePauliOp.from_list(
    [("ZIII", 1.0)]
)

estimator_test = StatevectorEstimator(
    seed=42
)

x_sample = (
    X_test_tensor[0]
    .detach()
    .cpu()
    .numpy()
)

quantum_weights = (
    model.quantum.weight
    .detach()
    .cpu()
    .numpy()
)

parameter_values = list(x_sample) + list(
    quantum_weights
)

# ============================================================
# DIRECT QISKIT ESTIMATOR REPRODUCIBILITY TEST
# ============================================================

job1 = estimator_test.run(
    [
        (
            qc,
            observable,
            [parameter_values]
        )
    ]
)

result1 = job1.result()

evs1 = result1[0].data.evs
value1 = evs1.item()


job2 = estimator_test.run(
    [
        (
            qc,
            observable,
            [parameter_values]
        )
    ]
)

result2 = job2.result()

evs2 = result2[0].data.evs
value2 = evs2.item()


print("=" * 60)
print("DIRECT ESTIMATOR TEST")
print("=" * 60)

print("EVS shape:", evs1.shape)

print("First result :", value1)
print("Second result:", value2)

print(
    "Difference:",
    abs(value1 - value2)
)

In [ ]:
# ============================================================
# DIRECT ESTIMATORQNN REPRODUCIBILITY TEST
# ============================================================

qnn = create_qnn(
    num_qubits=4
)

x_qnn = X_test_tensor[:10].clone().detach()

weights_qnn = (
    model.quantum.weight
    .detach()
    .clone()
)

# First QNN evaluation
output1 = qnn.forward(
    input_data=x_qnn,
    weights=weights_qnn
)

# Second QNN evaluation
output2 = qnn.forward(
    input_data=x_qnn,
    weights=weights_qnn
)

output1 = torch.as_tensor(output1)
output2 = torch.as_tensor(output2)

difference = torch.max(
    torch.abs(output1 - output2)
).item()

print("=" * 60)
print("DIRECT ESTIMATORQNN TEST")
print("=" * 60)

print("Output 1:")
print(output1)

print()

print("Output 2:")
print(output2)

print()

print("Maximum difference:", difference)

print(
    "Outputs identical:",
    torch.allclose(
        output1,
        output2,
        atol=1e-7,
        rtol=1e-7
    )
)

In [ ]:
# ============================================================
# INSPECT ESTIMATORQNN
# ============================================================

print("=" * 60)
print("ESTIMATORQNN CONFIGURATION")
print("=" * 60)

print("QNN:")
print(qnn)

print()
print("Estimator:")
print(qnn.estimator)

print()
print("Estimator type:")
print(type(qnn.estimator))

print()
print("Input parameters:")
print(list(qnn.input_params))

print()
print("Weight parameters:")
print(list(qnn.weight_params))


In [ ]:
print("=" * 60)
print("ESTIMATOR OPTIONS")
print("=" * 60)

print(qnn.estimator.options)

print("=" * 60)
print("ESTIMATOR SEED")
print("=" * 60)

print(
    "Seed:",
    getattr(qnn.estimator.options, "seed", None)
)

In [ ]:
# ============================================================
# EXPLICIT SEEDED ESTIMATORQNN
# ============================================================

from qiskit.primitives import StatevectorEstimator
from qiskit_machine_learning.neural_networks import EstimatorQNN

estimator_seeded = StatevectorEstimator(
    seed=42
)

qnn_seeded = EstimatorQNN(
    circuit=qc,
    estimator=estimator_seeded,
    observables=observable,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters,
)
# ============================================================
# TEST SEEDED ESTIMATORQNN
# ============================================================

output1 = qnn_seeded.forward(
    input_data=x_qnn,
    weights=weights_qnn
)

output2 = qnn_seeded.forward(
    input_data=x_qnn,
    weights=weights_qnn
)

output1 = torch.as_tensor(output1)
output2 = torch.as_tensor(output2)

difference = torch.max(
    torch.abs(output1 - output2)
).item()

print("=" * 60)
print("SEEDED ESTIMATORQNN TEST")
print("=" * 60)

print("Output 1:")
print(output1)

print()

print("Output 2:")
print(output2)

print()

print("Maximum difference:", difference)

print(
    "Outputs identical:",
    torch.allclose(
        output1,
        output2,
        atol=1e-7,
        rtol=1e-7
    )
)

In [ ]:
# ============================================================
# SEEDED ESTIMATORQNN REPRODUCIBILITY TEST
# ============================================================

from qiskit.primitives import StatevectorEstimator
from qiskit_machine_learning.neural_networks import EstimatorQNN

# Create a fresh seeded estimator
estimator_seeded = StatevectorEstimator(
    seed=42
)

# Create a fresh QNN
qnn_seeded = EstimatorQNN(
    circuit=qc,
    estimator=estimator_seeded,
    observables=observable,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters,
)

# Same inputs
x_qnn = X_test_tensor[:10].clone().detach()

weights_qnn = (
    model.quantum.weight
    .detach()
    .clone()
)

# ------------------------------------------------------------
# First forward pass
# ------------------------------------------------------------

output1 = qnn_seeded.forward(
    input_data=x_qnn,
    weights=weights_qnn
)

# ------------------------------------------------------------
# Second forward pass
# ------------------------------------------------------------

output2 = qnn_seeded.forward(
    input_data=x_qnn,
    weights=weights_qnn
)

# Convert to tensors
output1 = torch.as_tensor(output1)
output2 = torch.as_tensor(output2)

# Difference
difference = torch.max(
    torch.abs(output1 - output2)
).item()

print("=" * 60)
print("SEEDED ESTIMATORQNN TEST")
print("=" * 60)

print("Output 1:")
print(output1)

print()

print("Output 2:")
print(output2)

print()

print("Maximum difference:", difference)

print(
    "Outputs identical:",
    torch.allclose(
        output1,
        output2,
        atol=1e-7,
        rtol=1e-7
    )
)

In [10]:
# ============================================================
# FINAL MODEL DETERMINISM CHECK
# ============================================================

model.eval()

x_check = X_test_tensor[:100].clone().detach()

with torch.no_grad():

    output1 = model(x_check)

    output2 = model(x_check)

output_difference = torch.max(
    torch.abs(output1 - output2)
).item()

prediction_change = (
    (output1 >= 0) != (output2 >= 0)
).float().mean().item()

print("=" * 60)
print("FINAL MODEL DETERMINISM CHECK")
print("=" * 60)

print(
    "Maximum output difference:",
    output_difference
)

print(
    "Prediction change rate:",
    prediction_change
)

print(
    "Outputs identical:",
    torch.allclose(
        output1,
        output2,
        atol=1e-7,
        rtol=1e-7
    )
)

FINAL MODEL DETERMINISM CHECK
Maximum output difference: 0.0
Prediction change rate: 0.0
Outputs identical: True


In [11]:
import torch

def fgsm_attack(
    model,
    x,
    y,
    epsilon,
):
    """
    Generate FGSM adversarial examples for binary classification.
    """

    model.eval()

    # Make sure labels are [N]
    y = y.detach().clone().view(-1).float()

    # Fresh leaf tensor for input gradients
    x_adv = x.detach().clone()
    x_adv.requires_grad_(True)

    # --------------------------------------------------------
    # Forward pass
    # --------------------------------------------------------

    logits = model(x_adv).view(-1)

    # --------------------------------------------------------
    # Loss
    # --------------------------------------------------------

    criterion = torch.nn.BCEWithLogitsLoss()

    loss = criterion(
        logits,
        y
    )

    # --------------------------------------------------------
    # Backward pass
    # --------------------------------------------------------

    model.zero_grad(set_to_none=True)

    loss.backward()

    # --------------------------------------------------------
    # Input gradient
    # --------------------------------------------------------

    if x_adv.grad is None:
        raise RuntimeError(
            "Input gradient is None. "
            "The model is not producing gradients "
            "with respect to the input."
        )

    gradient = x_adv.grad.detach()

    # --------------------------------------------------------
    # FGSM perturbation
    # --------------------------------------------------------

    perturbation = epsilon * gradient.sign()

    x_adv = x_adv.detach() + perturbation

    return x_adv.detach(), perturbation.detach()

# ============================================================
# FGSM SANITY CHECK — EPSILON = 0
# ============================================================

model.eval()

epsilon = 0.0

x_clean = X_test_tensor.clone().detach()
y_true = y_test_tensor.clone().detach()

# ------------------------------------------------------------
# Clean predictions
# ------------------------------------------------------------

with torch.no_grad():
    clean_logits = model(x_clean)

clean_predictions = (
    clean_logits.squeeze(1) >= 0
).long()

clean_accuracy = (
    clean_predictions == y_true
).float().mean().item()

# ------------------------------------------------------------
# FGSM with epsilon = 0
# ------------------------------------------------------------

x_adv, perturbation = fgsm_attack(
    model=model,
    x=x_clean,
    y=y_true,
    epsilon=epsilon,
)

# ------------------------------------------------------------
# Adversarial predictions
# ------------------------------------------------------------

with torch.no_grad():
    adv_logits = model(x_adv)

adv_predictions = (
    adv_logits.squeeze(1) >= 0
).long()

adv_accuracy = (
    adv_predictions == y_true
).float().mean().item()

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

accuracy_drop = (
    clean_accuracy - adv_accuracy
)

prediction_change_rate = (
    clean_predictions != adv_predictions
).float().mean().item()

clean_correct = (
    clean_predictions == y_true
)

successful_attacks = (
    clean_correct &
    (adv_predictions != y_true)
)

attack_success_rate = (
    successful_attacks.sum().item()
    / max(clean_correct.sum().item(), 1)
)

mean_linf = (
    perturbation.abs()
    .max(dim=1)
    .values
    .mean()
    .item()
)

max_linf = (
    perturbation.abs()
    .max(dim=1)
    .values
    .max()
    .item()
)

print("=" * 60)
print("FGSM SANITY CHECK — EPSILON = 0")
print("=" * 60)

print(f"Clean accuracy:          {clean_accuracy:.10f}")
print(f"Adversarial accuracy:    {adv_accuracy:.10f}")
print(f"Accuracy drop:           {accuracy_drop:.10f}")
print(f"Prediction change rate:  {prediction_change_rate:.10f}")
print(f"Attack success rate:     {attack_success_rate:.10f}")
print(f"Mean L-inf perturbation: {mean_linf:.10f}")
print(f"Max L-inf perturbation:  {max_linf:.10f}")

FGSM SANITY CHECK — EPSILON = 0
Clean accuracy:          0.5034413338
Adversarial accuracy:    0.5034413338
Accuracy drop:           0.0000000000
Prediction change rate:  0.0000000000
Attack success rate:     0.0000000000
Mean L-inf perturbation: 0.0000000000
Max L-inf perturbation:  0.0000000000


In [12]:
# ============================================================
# INPUT GRADIENT TEST
# ============================================================

model.eval()

x_test_small = X_test_tensor[:8].clone().detach()
y_test_small = y_test_tensor[:8].clone().detach().view(-1).float()

x_test_small.requires_grad_(True)

logits = model(x_test_small).view(-1)

criterion = torch.nn.BCEWithLogitsLoss()

loss = criterion(
    logits,
    y_test_small
)

model.zero_grad(set_to_none=True)

loss.backward()

print("=" * 60)
print("INPUT GRADIENT TEST")
print("=" * 60)

print("Loss:", loss.item())

print("Input gradient is None:")
print(x_test_small.grad is None)

if x_test_small.grad is not None:

    print(
        "Gradient shape:",
        x_test_small.grad.shape
    )

    print(
        "Gradient min:",
        x_test_small.grad.min().item()
    )

    print(
        "Gradient max:",
        x_test_small.grad.max().item()
    )

    print(
        "Gradient mean:",
        x_test_small.grad.mean().item()
    )

    print(
        "Gradient norm:",
        x_test_small.grad.norm().item()
    )

INPUT GRADIENT TEST
Loss: 0.7398929595947266
Input gradient is None:
False
Gradient shape: torch.Size([8, 4])
Gradient min: -0.4062500298023224
Gradient max: 0.36549243330955505
Gradient mean: -0.0024851392954587936
Gradient norm: 0.9935846924781799


In [13]:
# ============================================================
# FGSM GRADIENT TEST
# ============================================================

x_small = X_test_tensor[:8].clone().detach()
y_small = y_test_tensor[:8].clone().detach()

x_adv_small, perturbation_small = fgsm_attack(
    model=model,
    x=x_small,
    y=y_small,
    epsilon=0.1,
)

print("=" * 60)
print("FGSM TEST — 8 SAMPLES")
print("=" * 60)

print("Original shape:")
print(x_small.shape)

print("Adversarial shape:")
print(x_adv_small.shape)

print("Perturbation shape:")
print(perturbation_small.shape)

print()

print("Maximum |perturbation|:")
print(
    perturbation_small.abs().max().item()
)

print("Mean |perturbation|:")
print(
    perturbation_small.abs().mean().item()
)

FGSM TEST — 8 SAMPLES
Original shape:
torch.Size([8, 4])
Adversarial shape:
torch.Size([8, 4])
Perturbation shape:
torch.Size([8, 4])

Maximum |perturbation|:
0.10000000149011612
Mean |perturbation|:
0.10000000894069672


In [14]:
# ============================================================
# FGSM SANITY CHECK — EPSILON = 0
# ============================================================

model.eval()

epsilon = 0.0

x_clean = X_test_tensor.clone().detach()
y_true = y_test_tensor.clone().detach().view(-1).long()

# ------------------------------------------------------------
# Clean predictions
# ------------------------------------------------------------

with torch.no_grad():
    clean_logits = model(x_clean).view(-1)

clean_predictions = (
    clean_logits >= 0
).long()

clean_accuracy = (
    clean_predictions == y_true
).float().mean().item()

# ------------------------------------------------------------
# FGSM epsilon = 0
# ------------------------------------------------------------

x_adv, perturbation = fgsm_attack(
    model=model,
    x=x_clean,
    y=y_true,
    epsilon=epsilon,
)

# ------------------------------------------------------------
# Adversarial predictions
# ------------------------------------------------------------

with torch.no_grad():
    adv_logits = model(x_adv).view(-1)

adv_predictions = (
    adv_logits >= 0
).long()

adv_accuracy = (
    adv_predictions == y_true
).float().mean().item()

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

accuracy_drop = (
    clean_accuracy - adv_accuracy
)

prediction_change_rate = (
    clean_predictions != adv_predictions
).float().mean().item()

clean_correct = (
    clean_predictions == y_true
)

successful_attacks = (
    clean_correct &
    (adv_predictions != y_true)
)

attack_success_rate = (
    successful_attacks.sum().item()
    / max(clean_correct.sum().item(), 1)
)

# ------------------------------------------------------------
# Perturbation statistics
# ------------------------------------------------------------

linf_per_sample = (
    perturbation.abs()
    .max(dim=1)
    .values
)

mean_linf = linf_per_sample.mean().item()
max_linf = linf_per_sample.max().item()

l2_per_sample = (
    perturbation
    .norm(p=2, dim=1)
)

mean_l2 = l2_per_sample.mean().item()
max_l2 = l2_per_sample.max().item()

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("=" * 60)
print("FGSM SANITY CHECK — EPSILON = 0")
print("=" * 60)

print(f"Clean accuracy:          {clean_accuracy:.10f}")
print(f"Adversarial accuracy:    {adv_accuracy:.10f}")
print(f"Accuracy drop:           {accuracy_drop:.10f}")
print(f"Prediction change rate:  {prediction_change_rate:.10f}")
print(f"Attack success rate:     {attack_success_rate:.10f}")
print(f"Mean L2 perturbation:    {mean_l2:.10f}")
print(f"Max L2 perturbation:     {max_l2:.10f}")
print(f"Mean L-inf perturbation: {mean_linf:.10f}")
print(f"Max L-inf perturbation:  {max_linf:.10f}")

FGSM SANITY CHECK — EPSILON = 0
Clean accuracy:          0.4979702234
Adversarial accuracy:    0.4979702234
Accuracy drop:           0.0000000000
Prediction change rate:  0.0000000000
Attack success rate:     0.0000000000
Mean L2 perturbation:    0.0000000000
Max L2 perturbation:     0.0000000000
Mean L-inf perturbation: 0.0000000000
Max L-inf perturbation:  0.0000000000
